# Long/Short MCP Dry-Run Portfolio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crowdvector/polybridge-cookbooks/blob/main/longshort-portfolio/longshort-portfolio.ipynb)

This notebook explains the Claude Desktop + PolyBridge MCP setup and then runs the reproducible REST-backed dry-run helper. It is dry-run only, review-only, requires human judgment, and is not financial advice.

PolyBridge MCP release: https://github.com/crowdvector/polybridge-search-mcp/releases/tag/polybridge-mcp-v0.2.4


In [ ]:
%pip install --quiet requests pillow


In [ ]:
from pathlib import Path
import sys

def resolve_workdir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "longshort-portfolio",
        Path.cwd().parent,
    ]
    for candidate in candidates:
        helper = candidate / "dry_run_portfolio.py"
        if helper.exists():
            return helper.parent
    raise FileNotFoundError("Could not locate longshort-portfolio/dry_run_portfolio.py")

workdir = resolve_workdir()
if str(workdir) not in sys.path:
    sys.path.insert(0, str(workdir))

import dry_run_portfolio
workdir


## Standardized Claude/MCP Prompt

The helper below uses REST for reproducibility. For the interactive workflow, use the exact prompt stored in `PROMPT.md`. If `POLYBRIDGE_API_KEY` is missing from the environment, notebook mode falls back to `getpass()` inside the helper instead of printing or persisting the key.


In [ ]:
from IPython.display import Markdown, display

display(Markdown((workdir / "PROMPT.md").read_text(encoding="utf-8")))


In [ ]:
paths, bundle = dry_run_portfolio.run_once(prompt_for_key=True)
paths.as_dict()


In [ ]:
from IPython.display import JSON, Image

display(Markdown("## Macro Snapshot"))
display(JSON(bundle["macro_snapshot"]))
display(Markdown("## Position Table"))
display(JSON(bundle["position_table"]))
display(Markdown("## Review-Only Orders"))
display(JSON(bundle["review_only_orders"]))

summary_png = paths.dry_run_summary_png
if summary_png is not None and summary_png.exists():
    display(Markdown("## Dry-Run Summary Image"))
    display(Image(filename=str(summary_png)))


In [ ]:
display(Markdown("## Research Brief"))
display(Markdown(bundle["research_brief"]))
display(Markdown("## Agent Session"))
display(Markdown(bundle["agent_session"]))
